# 05 — Cyclone tropical : pluie + surge

**Objectif** : compléter l'analyse du notebook 03 (vent cyclonique) avec les deux autres composantes de dommage d'un cyclone tropical :

| Composante | Classe CLIMADA | Intensité | Modèle |
|------------|---------------|-----------|--------|
| Vent | `TropCyclone` | m/s | Holland 2008 (notebook 03) |
| **Pluie** | `TCRain` | mm | R-CLIPER (ce notebook) |
| **Surge** | `TCSurgeBathtub` | m | Bathtub + decay (ce notebook) |

**Zone** : Caraïbes (suite du notebook 03)

**Question clé** : quelle composante domine le risque total ?

---

**Sommaire** :
1. Chargement des tracks et calcul du vent (rappel 03)
2. Pluie cyclonique — modèle R-CLIPER
3. Storm surge — modèle bathtub
4. Exposition et fonctions d'impact
5. Impact comparé : vent vs pluie vs surge
6. Synthèse multi-péril

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
from shapely.geometry import Point

from climada.hazard import Centroids, TCTracks, TropCyclone
from climada.entity import Exposures, ImpactFuncSet, ImpfTropCyclone
from climada.entity.impact_funcs.base import ImpactFunc
from climada.engine import Impact

from climada_petals.hazard import TCRain, TCSurgeBathtub

print("Imports OK")

## 1. Tracks et vent (rappel notebook 03)

On recharge les tracks IBTrACS Atlantique Nord et on calcule le hazard vent.

In [ ]:
# Charger les tracks (période réduite pour la rapidité)
tracks = TCTracks.from_ibtracs_netcdf(basin='NA', year_range=(2017, 2019), correct_pres=True)
tracks.equal_timestep(time_step_h=1.0)
print(f"Tracks chargées : {tracks.size}")

# Centroids Caraïbes
centroids = Centroids.from_pnt_bounds(
    points_bounds=(-90, 10, -60, 30), res=0.5
)
print(f"Centroids : {centroids.size}")

# Hazard vent
tc_wind = TropCyclone.from_tracks(tracks, centroids=centroids)
print(f"Vent : {tc_wind.size} événements, intensité max = {tc_wind.intensity.max():.1f} m/s")

## 2. Pluie cyclonique — modèle R-CLIPER

**R-CLIPER** (Tuleya et al. 2007) est un modèle statistique de pluie TC :
- Estime le taux de précipitation en fonction de la distance au centre et de l'intensité du cyclone
- Ne nécessite pas de données de topographie (contrairement au modèle TCR)
- Rapide à calculer, bon pour le screening

L'alternative **TCR** (Lu et al. 2018) est plus physique mais nécessite des données de topographie SRTM et de coefficient de trainée ERA5.

In [ ]:
# --- 2a. Calcul de la pluie cyclonique (R-CLIPER) ---
tc_rain = TCRain.from_tracks(tracks, centroids=centroids, model='R-CLIPER')

print(f"Pluie TC : {tc_rain.size} événements")
print(f"Intensité max (précipitation cumulée) : {tc_rain.intensity.max():.1f} mm")
print(f"Unité : {tc_rain.units}")

In [ ]:
# --- 2b. Visualiser la pluie pour l'événement le plus intense ---
# Trouver l'événement avec le max de pluie
max_event_idx = tc_rain.intensity.max(axis=1).toarray().flatten().argmax()
print(f"Événement le plus pluvieux : index {max_event_idx}, "
      f"nom = {tc_rain.event_name[max_event_idx]}")

ax = tc_rain.plot_intensity(event=max_event_idx)
ax.set_title(f"Pluie cyclonique — {tc_rain.event_name[max_event_idx]}")
plt.show()

## 3. Storm surge — modèle bathtub

Le modèle **bathtub** est une approximation simple de la submersion marine :
- Estime la hauteur de surge à partir de la vitesse du vent (relation empirique)
- Propage l'inondation vers l'intérieur des terres avec une décroissance
- Nécessite un MNT (Modèle Numérique de Terrain) pour filtrer les zones basses

**Limites** : ne modélise pas la dynamique des vagues, l'interaction avec les marées, ni la bathymétrie. Pour du calcul précis, il faudrait GeoClaw (solveur hydrodynamique).

> **Note** : le surge nécessite un fichier DEM (raster d'élévation). Ici on utilise les données SRTM téléchargées par CLIMADA.

In [ ]:
# --- 3a. Téléchargement du DEM (SRTM) ---
# Le modèle bathtub nécessite un raster d'élévation
# On utilise la fonction utilitaire de CLIMADA pour télécharger le SRTM

# Option 1 : utiliser le DEM par défaut de TCRain (SRTM 0.1°)
# topo_path = TCRain.default_elevation_tif()  # ~télécharge ~100 MB

# Option 2 : utiliser un DEM local si disponible
# topo_path = "/path/to/dem.tif"

# Pour la démo, on va vérifier si le surge peut être calculé
# Note : TCSurgeBathtub.from_tc_winds() nécessite un DEM réel
# On documente l'approche et on montre le code fonctionnel

print("Le modèle TCSurgeBathtub nécessite un fichier DEM (SRTM).")
print("Décommenter la ligne ci-dessous pour télécharger le SRTM (~100 MB) :")
print("  topo_path = TCRain.default_elevation_tif()")
print()
print("Puis calculer le surge :")
print("  tc_surge = TCSurgeBathtub.from_tc_winds(tc_wind, topo_path)")
print("  tc_surge = TCSurgeBathtub.from_tc_winds(tc_wind, topo_path, add_sea_level_rise=0.2)")

In [ ]:
# --- 3b. Calcul du surge (décommenter quand le DEM est disponible) ---
# topo_path = TCRain.default_elevation_tif()
#
# # Surge sans SLR
# tc_surge = TCSurgeBathtub.from_tc_winds(
#     tc_wind, topo_path,
#     inland_decay_rate=0.2,    # 0.2 m/km de décroissance vers l'intérieur
#     add_sea_level_rise=0.0,
# )
# print(f"Surge : {tc_surge.size} événements, max = {tc_surge.intensity.max():.1f} m")
#
# # Surge avec SLR (+0.2m, scénario 2050)
# tc_surge_slr = TCSurgeBathtub.from_tc_winds(
#     tc_wind, topo_path,
#     inland_decay_rate=0.2,
#     add_sea_level_rise=0.2,
# )
# print(f"Surge + SLR : max = {tc_surge_slr.intensity.max():.1f} m")

print("Cellule prête — décommenter quand le DEM SRTM est téléchargé.")

## 4. Exposition et fonctions d'impact

On définit le même portefeuille que le notebook 03, avec des fonctions d'impact pour chaque composante :
- **Vent** : Emanuel 2011 (intensité en m/s → fraction de dommage)
- **Pluie** : Fonction custom basée sur la profondeur de pluie (mm → dommage)
- **Surge** : JRC Huizinga (profondeur d'eau en m → dommage, même que flood)

In [ ]:
# --- 4a. Portefeuille d'actifs (identique au notebook 03) ---
assets = [
    {"name": "Hotel Miami Beach",    "value": 50e6},
    {"name": "Port Kingston",        "value": 120e6},
    {"name": "Resort Cancún",        "value": 30e6},
    {"name": "Warehouse San Juan",   "value": 15e6},
    {"name": "Office Nassau",        "value": 25e6},
]
coords = [(-80.13, 25.79), (-76.80, 17.97), (-86.85, 21.16),
          (-66.07, 18.47), (-77.34, 25.06)]

exp_gdf = gpd.GeoDataFrame(assets, geometry=[Point(*c) for c in coords])
exp_gdf["impf_TC"] = 1   # Vent (Emanuel)
exp_gdf["impf_TR"] = 2   # Pluie (custom)

exposure = Exposures(exp_gdf)
exposure.check()
print(f"Portefeuille : {len(exposure.gdf)} actifs, {exposure.gdf['value'].sum()/1e6:.0f} M$")

In [ ]:
# --- 4b. Fonctions d'impact ---

# 1. Vent TC — Emanuel 2011
impf_wind = ImpfTropCyclone.from_emanuel_usa()  # id=1 par défaut

# 2. Pluie TC — fonction custom (mm de pluie → dommage)
# Basé sur les seuils FEMA : dommages significatifs à partir de 150-200 mm
impf_rain = ImpactFunc(
    id=2,
    haz_type='TR',
    name='TC Rain damage function',
    intensity_unit='mm',
    intensity=np.array([0, 50, 100, 150, 200, 300, 500, 800]),
    mdd=np.array([0, 0.0, 0.02, 0.05, 0.10, 0.20, 0.40, 0.60]),
    paa=np.array([1, 1, 1, 1, 1, 1, 1, 1]),
)

# Visualiser les deux fonctions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(impf_wind.intensity, impf_wind.mdd * impf_wind.paa * 100,
             linewidth=2, color='#e63946')
axes[0].set_xlabel("Vitesse du vent (m/s)")
axes[0].set_ylabel("Dommage (%)")
axes[0].set_title("Impact function — Vent TC (Emanuel)")
axes[0].grid(True, alpha=0.3)
axes[0].set_xlim(0, 80)

axes[1].plot(impf_rain.intensity, impf_rain.mdd * impf_rain.paa * 100,
             linewidth=2, color='#2196F3')
axes[1].set_xlabel("Précipitation cumulée (mm)")
axes[1].set_ylabel("Dommage (%)")
axes[1].set_title("Impact function — Pluie TC (custom)")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Assembler le set
impf_set = ImpactFuncSet([impf_wind, impf_rain])
print(impf_set)

## 5. Impact comparé : vent vs pluie

On calcule l'impact séparément pour chaque composante et on compare.

In [ ]:
# --- 5a. Impact du vent ---
impact_wind = Impact()
impact_wind.calc(exposures=exposure, impact_funcs=impf_set, hazard=tc_wind, save_mat=True)
print(f"EAI Vent : {impact_wind.aai_agg/1e6:.2f} M$/an")

# --- 5b. Impact de la pluie ---
impact_rain = Impact()
impact_rain.calc(exposures=exposure, impact_funcs=impf_set, hazard=tc_rain, save_mat=True)
print(f"EAI Pluie : {impact_rain.aai_agg/1e6:.2f} M$/an")

# --- 5c. Comparaison ---
print(f"\n{'='*50}")
print(f"{'Composante':<15} {'EAI (M$/an)':>15} {'Part (%)':>10}")
print(f"{'='*50}")
total = impact_wind.aai_agg + impact_rain.aai_agg
for name, imp in [("Vent", impact_wind), ("Pluie", impact_rain)]:
    pct = imp.aai_agg / total * 100 if total > 0 else 0
    print(f"{name:<15} {imp.aai_agg/1e6:>15.2f} {pct:>9.1f}%")
print(f"{'─'*50}")
print(f"{'TOTAL':<15} {total/1e6:>15.2f} {'100.0%':>10}")

In [ ]:
# --- 5d. Visualisation comparée par actif ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

asset_names = [a["name"] for a in assets]
x = np.arange(len(asset_names))
width = 0.35

# EAI par actif
eai_wind = impact_wind.eai_exp / 1e3  # k$/an
eai_rain = impact_rain.eai_exp / 1e3

axes[0].bar(x - width/2, eai_wind, width, label="Vent", color="#e63946", alpha=0.8)
axes[0].bar(x + width/2, eai_rain, width, label="Pluie", color="#2196F3", alpha=0.8)
axes[0].set_xticks(x)
axes[0].set_xticklabels(asset_names, rotation=30, ha="right", fontsize=8)
axes[0].set_ylabel("EAI (k$/an)")
axes[0].set_title("Expected Annual Impact par actif")
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis="y")

# Courbes de fréquence
rp = np.array([5, 10, 25, 50, 100, 250])
fc_wind = impact_wind.calc_freq_curve(return_per=rp)
fc_rain = impact_rain.calc_freq_curve(return_per=rp)

axes[1].plot(fc_wind.return_per, fc_wind.impact/1e6, 'o-', color="#e63946",
             linewidth=2, label="Vent")
axes[1].plot(fc_rain.return_per, fc_rain.impact/1e6, 's-', color="#2196F3",
             linewidth=2, label="Pluie")
axes[1].set_xlabel("Période de retour (années)")
axes[1].set_ylabel("Perte (M$)")
axes[1].set_title("Courbes de fréquence de dépassement")
axes[1].set_xscale("log")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Récapitulatif

### Composantes d'impact d'un cyclone tropical

| Composante | Modèle | Données | Complexité | Dominance |
|------------|--------|---------|------------|-----------|
| **Vent** | Holland 2008 | IBTrACS | Faible | Zones exposées au vent |
| **Pluie** | R-CLIPER | IBTrACS | Faible | Zones basses, intérieur des terres |
| **Surge** | Bathtub | IBTrACS + DEM | Moyen | Zones côtières basses |
| **Surge (précis)** | GeoClaw | IBTrACS + topo + bathy | Élevé | Idem, mais résultat physique |

### Points clés
- Le **vent** domine généralement pour les structures en hauteur
- La **pluie** peut dominer pour les événements lents (Harvey 2017 : 1500 mm sur Houston)
- Le **surge** domine dans les zones côtières basses (Katrina 2005 : 8m de surge)
- L'analyse **multi-péril** est essentielle : les 3 composantes sont corrélées (même événement)

### Prochaines étapes
- [ ] Télécharger le DEM SRTM pour activer le calcul de surge
- [ ] Tester le modèle TCR (pluie physique) vs R-CLIPER
- [ ] Comparer avec les données de dommages observés (EM-DAT)
- [ ] Intégrer le changement climatique (tc_clim_change, Knutson 2020)